# Homework - Grover MaxCut

The places where you have enter code are marked with `# YOUR CODE HERE`.

In [65]:
import cirq
from cirq import H, X, Y, Z, CX, inverse

## Question 1 (4 points)

Write a function, `oracle010`, that implements an oracle that marks the state $|010 \rangle$. The function `oracle010` has

* input: `qq`, a 3-qubit register 
* returns: `None`

The function should append a sequence of gates to `qq` to mark the state $|010\rangle$ only. Don't append any measurements to `qq`.

To help you test the function, we have provided the `grover_diffusion` and `grover` functions.

In [66]:
from cirq import X, Z, H

def oracle010(qq):
    # YOUR CODE HERE
    yield X.on(qq[0])
    yield X.on(qq[2])

    yield Z.on(qq[2]).controlled_by(qq[0], qq[1])

    yield X.on(qq[0])
    yield X.on(qq[2])

In [67]:

# visualize your implemented gates
qqTest = cirq.LineQubit.range(3)
circuit = cirq.Circuit()
circuit.append(oracle010(qqTest))
circuit

0: ───X───@───X───
          │
1: ───────@───────
          │
2: ───X───@───X───

In [68]:
# To check your solution, we need some to implement grover
def grover_diffusion(qq,n):
    yield H.on_each(*qq)
    yield X.on_each(*qq)
    yield Z(qq[n-1]).controlled_by(*(qq[0:n-1]))
    yield X.on_each(*qq)
    yield H.on_each(*qq)

In [69]:
def grover(trials_number):
    n=3
    qq = cirq.LineQubit.range(n)
    circuit = cirq.Circuit()
    circuit.append(H.on_each(*qq))  

    for i in range(2):
        circuit.append(oracle010(qq))
        circuit.append(grover_diffusion(qq,n))
    circuit.append(cirq.measure(*qq, key='result'))

    # determine the statistics of the measurements
    s = cirq.Simulator() 
    samples = s.run(circuit, repetitions=trials_number)

    def bitstring(bits):
        return "".join(str(int(b)) for b in bits)

    counts = samples.histogram(key="result",fold_func=bitstring)
    print(counts)
    return counts.get('010')

In [70]:
# run grover to test if your function gives the right answer
grover(100)

Counter({'010': 92, '011': 2, '100': 2, '111': 2, '101': 1, '000': 1})


92

In [71]:
# hidden tests in this cell will be used for grading.

## Question 2 (6 points)

Graph $G$ has 5 vertices and 6 edges: (0,3), (0,4), (1,3), (1,4), (2,3), (2,4).

Write an oracle for the graph $G$ to check whether it admits a valid 2-coloring. 

The function `oracle2` has

* input: `qq`, a 12-qubit register 
* returns: `None`

The function should append only a sequence of gates to `qq`. It should not append any measurements to `qq`.

Use qubits 0-4 for the vertices, 5-10 for the edges and 11 as the ancilla. 

You can test the oracle with the provided `grover_diffusion`, `grover` and `oracle_computation2` functions.

In [72]:
import cirq

def oracle2(qq):
    aristas = [
        (0, 3, 5), (0, 4, 6),
        (1, 3, 7), (1, 4, 8),
        (2, 3, 9), (2, 4, 10)
    ]
    
    for u, v, edge_q in aristas:
        yield cirq.CX(qq[u], qq[edge_q])
        yield cirq.CX(qq[v], qq[edge_q])
        
    yield cirq.X(qq[11]).controlled_by(*(qq[5:11]))
    
    for u, v, edge_q in reversed(aristas):
        yield cirq.CX(qq[v], qq[edge_q])
        yield cirq.CX(qq[u], qq[edge_q])

In [73]:
# We need some code so you can check your solution
def oracle_computation2(qq):
    yield oracle2(qq)
    yield Z(qq[11])
    yield inverse(oracle2(qq))  

In [74]:
def grover2(trials_number):    
    import cirq
    from cirq import X, H, Z, inverse, CX
    s = cirq.Simulator()

    qq = cirq.LineQubit.range(12)
    n=5
    
    circuit = cirq.Circuit()
    circuit.append(H.on_each(*(qq[0:n])))
    for i in range(2):
        circuit.append(oracle_computation2(qq))
        circuit.append(grover_diffusion(qq,n))

    circuit.append(cirq.measure(*(qq[0:n]), key='result'))

    # determine the statistics of the measurements
    samples = s.run(circuit, repetitions=trials_number)
    result = samples.measurements["result"]

    def bitstring(bits):
        return "".join(str(int(b)) for b in bits)

    counts = samples.histogram(key="result",fold_func=bitstring)
    return counts

In [75]:
#You can use this cell to test your solution
shots=1000
grover2(shots)

Counter({'00011': 476,
         '11100': 437,
         '00010': 8,
         '11001': 7,
         '10100': 5,
         '00110': 4,
         '01110': 4,
         '00111': 4,
         '11110': 4,
         '11010': 4,
         '00001': 4,
         '11111': 3,
         '01100': 3,
         '01011': 3,
         '11101': 3,
         '10011': 3,
         '10111': 3,
         '01111': 3,
         '10001': 3,
         '01001': 2,
         '11000': 2,
         '00101': 2,
         '10000': 2,
         '00100': 2,
         '11011': 2,
         '01000': 2,
         '01010': 1,
         '10101': 1,
         '01101': 1,
         '10010': 1,
         '10110': 1})

In [76]:
# hidden tests in this cell will be used for grading.

## Question 3 (10 points)

Graph $G$ has 4 vertices and 5 edges: (0,1), (0,2), (0,3), (1,2), (1,3)

Write an oracle for the graph $G$ to check whether there exists a coloring with at least 4 edges connecting vertices with different colors.

The function `oracle3` has

* input: `qq`, a 13-qubit register 
* returns: `None`

The function should append only a sequence of gates to `qq`. It should not append any measurements to `qq`.

Use qubits 
- 0-3 for the vertices,
- 4-8 for the edges,
- 9-11 for the addition (remember we need three qubits here for addition unlike the last question), and
- 12 as the ancilla.

You can test the oracle with the provided `grover_diffusion`, `grover` and `oracle_computation3` functions.

In [77]:
import cirq
from cirq import CX, CCX, X, Z, inverse

def oracle3(qq):
    aristas = [(0, 1, 4), (0, 2, 5), (0, 3, 6), (1, 2, 7), (1, 3, 8)]
    
    for u, v, edge_q in aristas:
        yield CX(qq[u], edge_q)
        yield CX(qq[v], edge_q)
        yield CCX(edge_q, qq[9], qq[10])
        yield CX(edge_q, qq[9])
        
    yield CX(qq[11], qq[12])
    
    for u, v, edge_q in reversed(aristas):
        yield CX(edge_q, qq[9])
        yield CCX(edge_q, qq[9], qq[10])
        yield CX(qq[v], edge_q)
        yield CX(qq[u], edge_q)

In [78]:
# We need some code so you can check your solution
def oracle_computation3(qq):
    qc_oracle = cirq.Circuit(oracle3(qq))
    
    yield qc_oracle.all_operations()
    yield oracle3(qq)
    yield Z(qq[12])
    yield inverse(oracle3(qq))  

In [79]:
import cirq
from cirq import X, H, Z, inverse, CX, CCX
    
def grover3(trials_number):    
    s = cirq.Simulator()

    qq = cirq.LineQubit.range(13)
    n=4

    circuit = cirq.Circuit()
    circuit.append(H.on_each(*(qq[0:n])))
    for i in range(2):
        circuit.append(oracle_computation3(qq))
        circuit.append(grover_diffusion(qq,n))

    circuit.append(cirq.measure(*(qq[0:n]), key='result'))

    # determine the statistics of the measurements
    samples = s.run(circuit, repetitions=trials_number)
    result = samples.measurements["result"]

    def bitstring(bits):
        return "".join(str(int(b)) for b in bits)

    counts = samples.histogram(key="result",fold_func=bitstring)
    return counts

In [80]:
#You can use this cell to test your solution
shots=1000
grover3(shots)

AttributeError: 'int' object has no attribute 'dimension'

In [ ]:
# hidden tests in this cell will be used for grading.